In [1]:
from dotenv import load_dotenv, find_dotenv
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
import os
from llama_index.embeddings.openai import OpenAIEmbedding

# Load environment variables
load_dotenv(find_dotenv())  # Ensure environment variables are loaded

# Get the embedding model from environment variables
embed_model = OpenAIEmbedding(model_name = os.environ.get("EMBED_MODEL"))

# If the embed_model is not found, raise an error or use a default model
if embed_model is None:
    raise ValueError("EMBED_MODEL environment variable is not set!")

# Load documents
documents = SimpleDirectoryReader(input_files=["../data/Pmg_lds.md"]).load_data()

# Create an index using the embedding model
index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)

# Initialize a query engine from the index
query_engine = index.as_query_engine()


In [2]:
response = query_engine.query("how can i find people to teach?")
print(response)

You can find people to teach by working with the bishop and the ward council to identify and contact individuals who have recently had a baby, moved to the area, or experienced a death in the family. Additionally, you can look for opportunities to offer simple service, teach members about the message of the Restoration, hold member firesides, offer to teach family home evening, take individuals on a tour of the local meetinghouse, arrange meetings with the bishop, invite people to visit www.mormon.org, organize scripture study classes, teach English as a second language, invite individuals to attend seminary or institute, go from home to home or talk to people on the streets, use pass-along cards, DVDs, videos, and brochures, seek referrals from various sources, coordinate with the Church’s public affairs representatives, invite people to Church meetings and activities, and invite people to baptismal services.


### Evaluating Baseline

In [20]:
from trulens_eval import Tru

tru = Tru()
tru.reset_database()


In [21]:
import numpy as np
from trulens.apps.llamaindex import TruLlama
from trulens.core import Feedback
from trulens.providers.openai import OpenAI

# Initialize provider class
provider = OpenAI(model_engine="gpt-4o-mini")

# select context to be used in feedback. the location of context is app specific.

context = TruLlama.select_context(query_engine)

# Define a groundedness feedback function
f_groundedness = (
    Feedback(
        provider.groundedness_measure_with_cot_reasons, name="Groundedness"
    )
    .on(context.collect())  # collect context chunks into a list
    .on_output()
)

# Question/answer relevance between overall question and answer.
f_answer_relevance = Feedback(
    provider.relevance_with_cot_reasons, name="Answer Relevance"
).on_input_output()
# Question/statement relevance between question and each context chunk.
f_context_relevance = (
    Feedback(
        provider.context_relevance_with_cot_reasons, name="Context Relevance"
    )
    .on_input()
    .on(context)
    .aggregate(np.mean)
)

✅ In Groundedness, input source will be set to __record__.app.query.rets.source_nodes[:].node.text.collect() .
✅ In Groundedness, input statement will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Answer Relevance, input prompt will be set to __record__.main_input or `Select.RecordInput` .
✅ In Answer Relevance, input response will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Context Relevance, input question will be set to __record__.main_input or `Select.RecordInput` .
✅ In Context Relevance, input context will be set to __record__.app.query.rets.source_nodes[:].node.text .


In [22]:
tru_query_engine_recorder = TruLlama(
    query_engine,
    app_name="LlamaIndex_App",
    app_version="baseline",
    feedbacks=[f_groundedness, f_answer_relevance, f_context_relevance],
)

In [17]:
# import csv

# # Initialize an empty list to store the questions
# questions = []

# # Open the CSV file
# with open("../dataset_eval/20_dataset.csv", mode='r', encoding='utf-8') as file:
#     # Create a CSV reader object
#     csv_reader = csv.DictReader(file)
    
#     # Loop through the rows and extract the 'question' column
#     for idx, row in enumerate(csv_reader):
#         if idx < 20:  # Only take the first 20 rows
#             questions.append(row['question'])  # Store the 'question' value

# # Print the list of sample questions
# print(questions)


['What strategies can be used to make a message easy to understand when teaching?', 'What should teachers do with unfamiliar words to ensure their message is easy to understand?', 'What are some effective study techniques to enhance understanding and retention of material?', 'What is the purpose of using a study journal in your scripture study?', 'What is the importance of organizing and summarizing lesson plans for effective teaching?', 'What is the recommended method for highlighting key words when marking scriptures?', 'What is the significance of beginning study activities with a prayer?', 'What is the significance of the power of ordination in the context of missionary work?', 'What is the purpose of marking scriptures in relation to applying gospel teachings?', 'What is the significance of preaching the gospel according to President Lorenzo Snow?', 'What is the relationship between the Atonement and missionary work according to President Howard W. Hunter?', 'What are the blessing

In [23]:
import pandas as pd

# Read the CSV file and drop the 'Unnamed: 0' column
df = pd.read_csv("../dataset_eval/20_dataset.csv").drop(columns=['Unnamed: 0'])

# Extract the first 20 questions into a list
questions = df['question'][:20].tolist()

# Print the list of questions
print(questions)


['What strategies can be used to make a message easy to understand when teaching?', 'What should teachers do with unfamiliar words to ensure their message is easy to understand?', 'What are some effective study techniques to enhance understanding and retention of material?', 'What is the purpose of using a study journal in your scripture study?', 'What is the importance of organizing and summarizing lesson plans for effective teaching?', 'What is the recommended method for highlighting key words when marking scriptures?', 'What is the significance of beginning study activities with a prayer?', 'What is the significance of the power of ordination in the context of missionary work?', 'What is the purpose of marking scriptures in relation to applying gospel teachings?', 'What is the significance of preaching the gospel according to President Lorenzo Snow?', 'What is the relationship between the Atonement and missionary work according to President Howard W. Hunter?', 'What are the blessing

In [24]:
# or as context manager
for question in questions:
    with tru_query_engine_recorder as recording:
        response = query_engine.query(question)

In [25]:
display(response)

Response(response="Grant emphasizes the joy and spiritual benefits of sharing the gospel, highlighting the importance of being a witness of God through baptism. He stresses the need to live the gospel faithfully and set an example for family and friends, while also actively seeking opportunities to share the message of the restored gospel. On the other hand, Hunter underscores the significance of taking the gospel to all people, emphasizing that it is a crucial responsibility in this mortal life. He connects the Atonement with missionary work, stating that experiencing its blessings naturally leads to a concern for others' well-being and a desire to share the gospel.", source_nodes=[NodeWithScore(node=TextNode(id_='feec4ed4-8252-46cd-afc9-8b50cbfd04f8', embedding=None, metadata={'file_path': '../data/Pmg_lds.md', 'file_name': 'Pmg_lds.md', 'file_type': 'text/markdown', 'file_size': 555277, 'creation_date': '2024-08-30', 'last_modified_date': '2024-08-30'}, excluded_embed_metadata_keys=

In [26]:
from trulens.dashboard.display import get_feedback_result

last_record = recording.records[-1]
get_feedback_result(last_record, "Context Relevance")

,question,context,ret
0,What do Grant and Hunter say about sharing the gospel with diverse groups?,"Missionary Work (Page 98)\nMembers who share the gospel experience joy and have the Spirit of the Lord more abundantly. As we share the gospel, we appreciate how precious and meaningful it is to us, and we feel a greater love for God and others. The Lord commanded His followers to preach the gospel in all the world, giving every person the opportunity to accept or reject it. When people are baptized, they make a covenant to always stand as witnesses of God. They are commanded to share the gospel with those who have not yet received it. As they live the gospel faithfully, they will set an example, showing their family members and friends the great blessings that come from living the gospel. They should also take advantage of opportunities to answer questions, share printed or audiovisual materials, and invite others to learn more about the message of the restored gospel. Members should pray for those who are not members of the Church. They should pray for missionary opportunities—to serve those who are not of our faith and share what they believe. The Lord promises to help members know what to say and do as they share the gospel.",0.333333
1,What do Grant and Hunter say about sharing the gospel with diverse groups?,"President Howard W. Hunter (1994–1995) (Page 27)\n“Surely taking the gospel to every kindred, tongue, and people is the single greatest responsibility we have in mortality. . . . We have been privileged to be born in these last days, as opposed to some earlier dispensation, to help take the gospel to all the earth” (“Walls of the Mind,” Ensign, Sept. 1990, 10).\n\n“What does the Atonement have to do with missionary work? Any time we experience the blessings of the Atonement in our lives, we cannot help but have a concern for the welfare of others. . . . A great indicator of one’s personal conversion is the desire to share the gospel with others.” (“The Atonement and Missionary Work,” seminar for new mission presidents, June 1994).",0.666667


In [27]:
from trulens.dashboard.display import get_feedback_result

last_record = recording.records[-1]
get_feedback_result(last_record, "Answer Relevance")

,prompt,response,ret
0,What do Grant and Hunter say about sharing the gospel with diverse groups?,"Grant emphasizes the joy and spiritual benefits of sharing the gospel, highlighting the importance of being a witness of God through baptism. He stresses the need to live the gospel faithfully and set an example for family and friends, while also actively seeking opportunities to share the message of the restored gospel. On the other hand, Hunter underscores the significance of taking the gospel to all people, emphasizing that it is a crucial responsibility in this mortal life. He connects the Atonement with missionary work, stating that experiencing its blessings naturally leads to a concern for others' well-being and a desire to share the gospel.",1.0


In [28]:
records, feedback = tru.get_records_and_feedback(app_ids=[])
# records.head()

In [29]:
import pandas as pd

pd.set_option("display.max_colwidth", None)
records[["input", "output"] + feedback]

,input,output,Context Relevance,Answer Relevance,Groundedness
0,"""What strategies can be used to make a message easy to understand when teaching?""","""To make a message easy to understand when teaching, one can simplify the language used, define unfamiliar words, use relatable examples or analogies, encourage questions for clarification, and ensure that the material is presented in a logical and organized manner.""",0.833333,1.000000,0.666667
1,"""What should teachers do with unfamiliar words to ensure their message is easy to understand?""","""Teachers should study key definitions and terms in the lesson materials, become familiar with the definitions provided, and use resources like True to the Faith and the Bible Dictionary to find definitions for other unfamiliar words. They should write these definitions in their study journal, share them with others to ensure understanding, and revise definitions if they are unclear. Teachers should take time to explain unfamiliar words or principles to ensure that their message is easy to understand and teach for understanding so that both they and those they teach can be edified and rejoice together.""",1.000000,1.000000,1.000000
2,"""What are some effective study techniques to enhance understanding and retention of material?""","""Some effective study techniques to enhance understanding and retention of material include exercising, showering, and praying before studying, studying at a desk or table, studying by topic, visualizing the material being studied, writing down questions and using study resources to find answers, sharing what you learn with others, marking scriptures and making notes, memorizing key passages, and actively engaging with the material by asking questions about the central message and how it applies personally.""",0.833333,1.000000,1.000000
3,"""What is the purpose of using a study journal in your scripture study?""","""The purpose of using a study journal in your scripture study is to help you understand, clarify, and remember what you are learning. It allows you to record and organize your thoughts and impressions in a way that fits how you learn, enabling you to easily access key information in the future. Additionally, reviewing your study journal helps you recall spiritual experiences, gain new insights, and recognize your personal growth.""",1.000000,1.000000,1.000000
4,"""What is the importance of organizing and summarizing lesson plans for effective teaching?""","""Organizing and summarizing lesson plans is important for effective teaching because it helps teachers present the doctrine clearly and with spiritual power. By focusing on the doctrine, baptismal interview questions, and commitments of the lessons, teachers can structure their teaching in a way that is organized, coherent, and impactful. This process also allows for the inclusion of scriptures, questions, and examples that enhance the lesson, making it more engaging and relevant to the needs of the individuals being taught. Additionally, continually refining and modifying lesson plans based on the needs of the learners ensures that the teaching approach is tailored to effectively meet those needs, ultimately leading to a more successful teaching experience.""",0.833333,1.000000,0.888889
5,"""What is the recommended method for highlighting key words when marking scriptures?""","""Circle or underline key words, and then use straight lines to link closely related words.""",1.000000,1.000000,1.000000
6,"""What is the significance of beginning study activities with a prayer?""","""Beginning study activities with a prayer is significant because it can help make study time more rewarding and spiritually uplifting. It sets a reverent tone, invites divine guidance, and helps individuals focus on building their faith in the Savior. By starting with a prayer, individuals can seek assistance in using their study time productively and in a way that benefits not only themselves but also others they may be 

In [30]:
tru.get_leaderboard()

,,Context Relevance,Answer Relevance,Groundedness,latency,total_cost
app_name,app_version,,,,,
LlamaIndex_App,baseline,0.85,0.983333,0.934722,3.95,0.001156


### Improving retriever